# Sentinel-2 optical comparison

The notebook source lives in this repository. Execute training on a Kaggle GPU. Use the official split lists and masks attached from the same dataset as the radar run. Fetch the matching optical chips directly from the original public Sen1Floods11 bucket. Fixed RGB and near-infrared preprocessing uses reflectance divided by 10,000 and clipped to [0, 1]. Ignore missing optical pixels and unknown labels. Cloud-obscured pixels can still have nonzero reflectance. Metrics apply to the optical valid-pixel population.

In [ ]:
import subprocess
import sys
from pathlib import Path

working = Path('/kaggle/working')
if not working.is_dir():
    raise RuntimeError('Execute this GPU training run on Kaggle.')
checkout = working/'source'
if not checkout.exists():
    subprocess.run(['git', 'clone', 'https://github.com/okiv21/Flood-Risk-Mapper.git', str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', 'f1df2d8b681dd6b04d690ed7a96011c15ad60483'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.6.0', 'rasterio'], check=True)
sys.path.insert(0, str(checkout))

In [ ]:
import json
from functools import partial
import pandas as pd
import torch
from torch.utils.data import DataLoader
from floodrisk.datasets.optical import OpticalChips, prepare
from floodrisk.models.unet import build_unet
from floodrisk.train import train, evaluate

if not torch.cuda.is_available():
    raise RuntimeError('Select the GPU accelerator before training.')
matches = list(Path('/kaggle/input').rglob('flood_train_data.csv'))
if len(matches) != 1:
    raise ValueError('Attach one hand-labelled dataset containing the official split lists.')
root = matches[0].parents[1]
cache = Path('/kaggle/temp/S2Hand')
prepare(root, cache)
dataset = partial(OpticalChips, cache=cache)
weights = working/'best_s2.pt'
history = train(root, out=str(weights), dataset=dataset, in_channels=4)
pd.DataFrame(history).to_csv(working/'history_s2.csv', index=False)

In [ ]:
model = build_unet(in_channels=4, pretrained=False).cuda()
model.load_state_dict(torch.load(weights, map_location='cuda', weights_only=True))
results = {split: evaluate(model, DataLoader(dataset(root, split), batch_size=8), 'cuda', threshold=0.5)
           for split in ['valid', 'test', 'bolivia']}
(working/'evaluation_s2.json').write_text(json.dumps(results, indent=2))
print(results)